<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 2 Exercise solutions

Packages that are being used in this notebook:

In [2]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.5.0+cu124
tiktoken version: 0.8.0


# Exercise 2.1

In [3]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [16]:
integers = tokenizer.encode("Everest")
print(integers)

for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")
    
tokenizer.decode(integers)

[36, 303, 2118]
36 -> E
303 -> ve
2118 -> rest


'Everest'

In [9]:
tokenizer.encode("Ak")

[33901]

In [6]:
tokenizer.encode("w")

[86]

In [7]:
tokenizer.encode("ir")

[343]

In [8]:
tokenizer.encode("w")

[86]

In [9]:
tokenizer.encode(" ")

[220]

In [10]:
tokenizer.encode("ier")

[959]

In [5]:
tokenizer.decode(integers)

'Akwirw ier'

# Exercise 2.2

In [2]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size)

    return dataloader


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

vocab_size = 50257
output_dim = 256
max_len = 4
context_length = max_len

token_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [3]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=2, stride=2)

for batch in dataloader:
    x, y = batch
    break

raw_text[:50], x, y, [tokenizer.decode(i) for i in x.tolist()], [tokenizer.decode(i) for i in y.tolist()]

('I HAD always thought Jack Gisburn rather a cheap g',
 tensor([[  40,  367],
         [2885, 1464],
         [1807, 3619],
         [ 402,  271]]),
 tensor([[  367,  2885],
         [ 1464,  1807],
         [ 3619,   402],
         [  271, 10899]]),
 ['I H', 'AD always', ' thought Jack', ' Gis'],
 [' HAD', ' always thought', ' Jack G', 'isburn'])

In [4]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=8, stride=8)

for batch in dataloader:
    x, y = batch
    break

raw_text[:50], x, y, [tokenizer.decode(i) for i in x.tolist()], [tokenizer.decode(i) for i in y.tolist()]

('I HAD always thought Jack Gisburn rather a cheap g',
 tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
         [10899,  2138,   257,  7026, 15632,   438,  2016,   257],
         [  922,  5891,  1576,   438,   568,   340,   373,   645],
         [ 1049,  5975,   284,   502,   284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632,   438,  2016,   257,   922],
         [ 5891,  1576,   438,   568,   340,   373,   645,  1049],
         [ 5975,   284,   502,   284,  3285,   326,    11,   287]]),
 ['I HAD always thought Jack Gis',
  'burn rather a cheap genius--though a',
  ' good fellow enough--so it was no',
  ' great surprise to me to hear that,'],
 [' HAD always thought Jack Gisburn',
  ' rather a cheap genius--though a good',
  ' fellow enough--so it was no great',
  ' surprise to me to hear that, in'])